In [1]:
import requests
import os
import json
import pandas as pd
from dotenv import load_dotenv, dotenv_values
from requests.exceptions import HTTPError, ConnectionError, Timeout, RequestException
from pathlib import Path
import sys

# --- portable paths --------------------------------------------------------
# Every file location in this notebook comes from paths.py, never from the
# working directory. This notebook sits in the project root next to paths.py,
# but the bootstrap below makes it run from ANY cwd (Jupyter, terminal, IDE).
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "paths.py").is_file()), None)
if ROOT is None:
    raise RuntimeError(
        "Cannot find paths.py in any parent of "
        f"{Path.cwd()} — run this notebook from inside the project folder.")
sys.path.insert(0, str(ROOT))
from paths import (ENV_FILE, CH_DATA, CHARGE_JSON_DIR,
                   COMPANIES_CSV, CHARGES_CSV, FILINGS_CSV, CLIENT_INPUT)
print(f"project root: {ROOT}")


project root: /Users/natchalin_/Projects/final_project/Lloyds


In [2]:
load_dotenv(ENV_FILE)

True

In [3]:
URL_BASE = "https://api.company-information.service.gov.uk"
URL_META_DATA = "https://document-api.company-information.service.gov.uk/document"
TIMEOUT = 10  # seconds

In [4]:
#Company House

def get(endpoint, params=None, headers=None):
    url = f"{URL_BASE}/{endpoint}"
    response = requests.get(
        url,
        auth=(os.getenv("CH_API"),""),
        headers=headers,
        params=params,
        timeout=TIMEOUT
        )
    response.raise_for_status()
    print(response.status_code)
    return response.json()

In [5]:
print(f"{ENV_FILE}   exists={ENV_FILE.exists()}")

/Users/natchalin_/Projects/final_project/Lloyds/.env   exists=True


In [6]:
import time, threading, itertools, re
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm      

# --- Multi-key throttled Companies House client (uses all API keys -> ~Nx faster)

def _clean_key(v):
    """Pull out the cleaned keys"""
    v = v.strip()
    m = re.search(r"""['"]([^'"]+)['"]""", v)   # value inside the first quote pair
    return m.group(1) if m else v.rstrip(",").strip()


def _load_ch_keys(names = ('CH_API', 'CH_API_2', 'CH_API_3', 'CH_API_4')):
    """Read the CH API keys out of .env, in `names` order.

    Parses the file directly rather than trusting os.getenv, so it works whether
    or not load_dotenv succeeded. Tolerates `KEY=v`, `KEY = "v",` and comment or
    blank lines, and ignores any non-CH key that may live in the same file.
    """
    found = {}
    for line in ENV_FILE.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue                       # comment / blank / junk -> skip
        k, v = line.split('=', 1)
        found[k.strip()] = _clean_key(v)

    parsed = [found[n] for n in names if found.get(n)]
    missing = [n for n in names if not found.get(n)]
    if missing:
        print(f"NOTE: {ENV_FILE} has no value for {missing} -> running on "
              f"{len(parsed)} key(s)")
    return parsed


_KEYS = _load_ch_keys()
if not _KEYS:
    raise RuntimeError("No Companies House API keys found (checked .env and env vars)")

_MIN_INTERVAL = 0.6           # seconds between calls PER KEY (~1.7/s; CH cap 2.0/s each)
MAX_WORKERS   = 2 * len(_KEYS)
print(f"{len(_KEYS)} API key(s) loaded -> up to ~{len(_KEYS) / _MIN_INTERVAL:.1f} req/s "
      f"with {MAX_WORKERS} workers")


class _Lane:
    """One API key: dedicated Session + a lock enforcing min-interval pacing."""
    __slots__ = ("session", "lock", "last")
    def __init__(self, key):
        self.session = requests.Session(); self.session.auth = (key, "")
        self.lock = threading.Lock(); self.last = 0.0
    def pace(self):
        with self.lock:
            wait = _MIN_INTERVAL - (time.monotonic() - self.last)
            if wait > 0:
                time.sleep(wait)
            self.last = time.monotonic()


_LANES = [_Lane(k) for k in _KEYS]
_rr = itertools.count(); _rr_lock = threading.Lock()

def _next_lane():
    with _rr_lock:
        i = next(_rr)
    return _LANES[i % len(_LANES)] # Return lane number (0 - 3) given the next count on the _rr (ticket dispenser)


def ch_get(path, params=None, max_retries=5):
    """Thread-safe, key-rotating, throttled GET. Returns a Response (200/404/etc.
       for the caller to interpret) or None if it gave up. Handles 429 (waits out
       the window) and transient network/5xx errors with exponential backoff."""
    url = f"{URL_BASE}{path}"
    for attempt in range(max_retries):
        lane = _next_lane()
        lane.pace()
        try:
            r = lane.session.get(url, params=params, timeout=TIMEOUT)
        except (ConnectionError, Timeout):
            time.sleep(2 ** attempt); continue
        if r.status_code == 429:                       # rare with per-key pacing
            time.sleep(int(r.headers.get("Retry-After", 60)) + 1); continue
        if r.status_code >= 500:                        # transient server error
            time.sleep(2 ** attempt); continue
        return r
    return None


def parallel_fill(df, todo_index, fn, cols, out_path,
                  desc="fetch", max_workers=None, checkpoint_every=200):
    """Concurrently apply fn(com_num) over todo_index rows and write results into
       `cols` (str, or list matching fn's tuple return). Only network I/O runs in
       threads; all DataFrame writes + checkpoints happen on the main thread, so no
       df locking is needed. Resumable + checkpointed + live rate/ETA."""
    cols = [cols] if isinstance(cols, str) else list(cols)
    todo_index = list(todo_index)
    n_todo = len(todo_index)
    workers = max_workers or MAX_WORKERS
    print(f"{n_todo} to fetch (of {len(df)}) — {len(_LANES)} keys x {workers} workers")
    t0 = time.time(); done = 0
    with ThreadPoolExecutor(max_workers=workers) as ex:
        fut2idx = {ex.submit(fn, df.at[idx, "com_num"]): idx for idx in todo_index}
        for fut in tqdm(as_completed(fut2idx), total=n_todo, desc=desc):
            idx = fut2idx[fut]
            try:
                res = fut.result()
            except Exception:
                res = "error" if len(cols) == 1 else tuple("error" for _ in cols)
            if len(cols) == 1:
                df.at[idx, cols[0]] = res
            else:
                for c, v in zip(cols, res):
                    df.at[idx, c] = v
            done += 1
            if done % checkpoint_every == 0:
                df.to_csv(out_path, index=False)
                rate = done / (time.time() - t0)
                eta = (n_todo - done) / rate / 60 if rate else 0
                print(f"  {done}/{n_todo} | {rate:.2f}/s | ETA ~{eta:.0f} min | checkpoint saved")
    if n_todo:                       # nothing fetched -> nothing changed -> skip the write
        df.to_csv(out_path, index=False)
        print(f"Saved -> {out_path}")
    else:
        print(f"Nothing to fetch — {out_path} left untouched.")

4 API key(s) loaded -> up to ~6.7 req/s with 8 workers


/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# --- Stage 0: ingest client lists -> company_raw/ --------------------------------
# Every CSV directly in client/input/ (NON-recursive: Template/ and Archive/ are
# deliberately skipped). Writes one column, com_num, which is all Stage 1 needs --
# it fetches sic_code and post_code from the API, so a bare list of company
# numbers is sufficient.
RAW_DIR = CH_DATA / "company_raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
CLIENT_PULL_FILE = RAW_DIR / "client_lists.csv"

client_nums, seen_files = set(), []
for f in sorted(CLIENT_INPUT.glob("*.csv")) if CLIENT_INPUT.is_dir() else []:
    one = pd.read_csv(f, dtype=str).dropna(how="all")     # drop Excel's blank rows
    col = next((c for c in one.columns if c.lower().strip() == "company_number"), None)
    if col is None:
        print(f"  SKIP {f.name}: no 'company_number' column ({list(one.columns)})")
        continue
    nums = (one[col].dropna().astype(str).str.strip().str.upper().str.zfill(8))
    client_nums |= set(nums)
    seen_files.append(f.name)
    print(f"  {f.name}: {len(one):,} rows -> {nums.nunique():,} unique company numbers")

if not client_nums:
    print(f"no client CSVs in {CLIENT_INPUT} -> nothing to ingest "
          "(training-only run; Stage 1 will use the bulk files as usual)")
else:
    pd.DataFrame({"com_num": sorted(client_nums)}).to_csv(CLIENT_PULL_FILE, index=False)
    already = set()
    if COMPANIES_CSV.exists():
        already = set(pd.read_csv(COMPANIES_CSV, dtype=str, usecols=["com_num"],
                                  low_memory=False)["com_num"]
                      .astype(str).str.strip().str.upper().str.zfill(8))
    new_n = len(client_nums - already)
    print(f"\n{len(client_nums):,} unique company numbers from {len(seen_files)} file(s)"
          f" -> {CLIENT_PULL_FILE.name}")
    print(f"  already held : {len(client_nums) - new_n:,}  (no API calls needed)")
    print(f"  NEW to pull  : {new_n:,}  (~{new_n * 3:,} API calls across Stages 1 + 3)")


  Test_3.csv: 99 rows -> 80 unique company numbers

80 unique company numbers from 1 file(s) -> client_lists.csv
  already held : 34  (no API calls needed)
  NEW to pull  : 46  (~138 API calls across Stages 1 + 3)


In [8]:
# --- Stage 1: ONE profile call per NEW company -> 8 columns ---------------------
# sic_code and post_code are pulled from the SAME profile call (zero extra cost).
# They used to come only from the raw CSVs, which meant a client sending a bare
# list of company numbers produced rows with no sector and no region -- two of the
# model's three categorical features. Reading them here makes Stage 1 self-sufficient.

BASE        = CH_DATA
RAW_DIR     = BASE / "company_raw"

COMPANIES   = COMPANIES_CSV
SME_TYPES   = ["small", "micro-entity", "total-exemption-full",
               "total-exemption-small", "medium"]

PROFILE_COLS = ["account_type", "accounts_overdue",
                "date_of_creation", "company_status",
                "last_accounts_made_up_to", "next_accounts_due_on",
                "sic_code", "post_code", "name"]

# 1. Combine every raw file; one row per company (raw files repeat com_num per SIC).
raw = pd.concat([pd.read_csv(f, dtype=str) for f in sorted(RAW_DIR.glob("*.csv"))],
                ignore_index=True).drop_duplicates("com_num")

# 2. Load prior results; NEW companies = those not already processed.
if COMPANIES.exists():
    done = pd.read_csv(COMPANIES, dtype=str)
else:
    done = pd.DataFrame(columns=list(raw.columns))
for c in PROFILE_COLS:
    if c not in done.columns:
        done[c] = pd.NA          

new = raw[~raw["com_num"].isin(done["com_num"])].copy() # Create a copy rows that 'raw' is not in 'done'

for c in PROFILE_COLS:
    if c not in new.columns:
        new[c] = pd.NA          # placeholder; the API fills it below
    # if the raw file already carries the column (sic_code / post_code), keep the
    # value as a fallback -- a successful API call overwrites it either way.

df = pd.concat([done, new], ignore_index=True)   # keep old results, append new rows

df = df.drop(columns=[c for c in ["com_type", "town", "county"]  # unused raw passthrough
                      if c in df.columns])


def fetch_profile_fields(company_num):
    """All nine columns from ONE profile call.
       Sentinels: 'no_accounts', 'not_found', 'error', 'unknown', 'none'."""
    r = ch_get(f"/company/{str(company_num).strip()}")
    if r is None:              return ("error",) * 9 # 9 columns -> need to show 9 errors
    if r.status_code == 404:   return ("not_found",) * 9
    if r.status_code != 200:   return ("error",) * 9
    j = r.json()
    acc = j.get("accounts", {})
    la  = acc.get("last_accounts", {})
    ov  = acc.get("overdue")
    return (la.get("type") or "no_accounts",
            "unknown" if ov is None else str(ov),
            j.get("date_of_creation") or "unknown",
            j.get("company_status") or "unknown",
            la.get("made_up_to") or "none",
            acc.get("next_accounts", {}).get("due_on") or "none",
            # sic_codes is a LIST; take the first, matching the raw files' convention
            # of one row per company after drop_duplicates("com_num").
            (j.get("sic_codes") or ["unknown"])[0],
            j.get("registered_office_address", {}).get("postal_code") or "unknown",
            # name came from the bulk raw files, which carry a `name` column --
            # a client list of bare company numbers does not, so fetch it here.
            j.get("company_name") or "unknown")


# Fetch rows missing account_type (new companies + any interrupted ones) OR
# missing name -- the latter self-heals the companies ingested from a client list
# before `name` was part of the profile pull. One call fills every column.
todo = df.index[df["account_type"].isna() | df["name"].isna()]
print(f"{len(new)} new companies from raw | {len(done)} already done")
parallel_fill(df, todo, fetch_profile_fields, PROFILE_COLS, COMPANIES, desc="profile")

# the SME flag replaces the old separate SME file — recomputed every run so a
# corrected account_type immediately moves a company in or out of the population
df["is_sme"] = df["account_type"].isin(SME_TYPES)
df.to_csv(COMPANIES, index=False)
print(f"\ncompanies.csv: {len(df):,} rows | is_sme {int(df['is_sme'].sum()):,} "
      f"| non-SME cached {int((~df['is_sme']).sum()):,}")
df["account_type"].value_counts(dropna=False)

46 new companies from raw | 839348 already done
137 to fetch (of 839394) — 4 keys x 8 workers


profile: 100%|██████████| 137/137 [00:20<00:00,  6.60it/s]


Saved -> /Users/natchalin_/Projects/final_project/Lloyds/API/CompaniesHouse/company_data/companies.csv

companies.csv: 839,394 rows | is_sme 567,932 | non-SME cached 271,462


account_type
micro-entity                   327524
total-exemption-full           227918
no_accounts                    208586
unaudited-abridged              28647
full                            12268
dormant                         11360
small                           11055
audit-exemption-subsidiary       6084
group                            4097
medium                           1185
total-exemption-small             250
audited-abridged                  176
null                               88
filing-exemption-subsidiary        69
no-accounts-type-available         55
error                              27
interim                             2
partial-exemption                   2
not_found                           1
Name: count, dtype: int64

In [9]:
# --- Remove perfect duplicate rows from the company table -----------------------
# A "perfect duplicate" = every column identical. The pull already dedupes on
# com_num, so this is now a harmless safety net (expect 0 removed). Operates on
# the WHOLE table (SME + cached non-SME) and writes every row back.
dedup = pd.read_csv(COMPANIES, dtype=str)
before = len(dedup)
dedup = dedup.drop_duplicates()          # all columns must match
removed = before - len(dedup)

dedup.to_csv(COMPANIES, index=False)
print(f"Removed {removed} perfect duplicate row(s): {before} -> {len(dedup)} rows in {COMPANIES}")


Removed 0 perfect duplicate row(s): 839394 -> 839394 rows in /Users/natchalin_/Projects/final_project/Lloyds/API/CompaniesHouse/company_data/companies.csv


In [10]:
# --- Stage 2: mark the SME population (no API calls, no second file) -------------
# Previously this REBUILT a separate SME csv from the acctype file, which silently
# dropped any column not listed in CARRY_COLS (that is how `region` kept vanishing).
# With one table there is nothing to rebuild: we just set a flag, so no column can
# ever be lost.
SME_TYPES = ["small", "micro-entity", "total-exemption-full",
             "total-exemption-small", "medium"]
COMPANIES = COMPANIES_CSV

# columns the enrichment stages fill in (created lazily so later stages can assume them)
ENRICHED_COLS = ["lloyds_customer", "recent_psc_kind", "recent_charge_status",
                 "recent_charge_created_on", "n_charges", "n_filings",
                 "region", "town_api"]

df = pd.read_csv(COMPANIES, dtype=str, low_memory=False)
df["is_sme"] = df["account_type"].isin(SME_TYPES)
for c in ENRICHED_COLS:
    if c not in df.columns:
        df[c] = pd.NA
df.to_csv(COMPANIES, index=False)

print(f"companies.csv: {len(df):,} rows")
print(f"  is_sme (modelling population): {int(df['is_sme'].sum()):,}")
print(f"  non-SME (kept as re-pull cache): {int((~df['is_sme']).sum()):,}")
print("\naccount types outside the SME set:")
print(df.loc[~df['is_sme'], 'account_type'].value_counts().head(6).to_string())

companies.csv: 839,394 rows
  is_sme (modelling population): 567,932
  non-SME (kept as re-pull cache): 271,462

account types outside the SME set:
account_type
no_accounts                   208586
unaudited-abridged             28647
full                           12268
dormant                        11360
audit-exemption-subsidiary      6084
group                           4097


In [11]:
import re

# --- Stage 3: SME enrichment (multi-key concurrent) ------------------------------
# ONE charges call (paged) + ONE PSC call per company fill FOUR columns:
#   lloyds_customer            <- charges: any charge held by a Lloyds-group entity
#   recent_charge_status       <- charges: most recent charge (latest created_on)
#   recent_charge_created_on   <- charges
#   recent_psc_kind            <- PSC: kind of the PSC with latest notified_on
# (accounts_overdue comes free with the stage-1 profile pull, not here.)
# Resumable: only processes rows still missing ANY of the four.
#
# Lloyds matching: the token "Lloyds" (no apostrophe) is the bank; it deliberately
# does NOT match "Lloyd's" (apostrophe) = Lloyd's of London, the insurance market.
# List covers the whole Lloyds Banking Group; trim to r"\blloyds\b" for bank-only.
LLOYDS_PATTERNS = [
    r"\blloyds\b",                       # Lloyds Bank, Lloyds TSB, Lloyds Commercial Finance, Lloyds Dev. Capital
    # NOTE the lookbehind: a bare "bank of scotland" also matches "The ROYAL Bank of
    # Scotland" (NatWest Group), which mislabelled 14,305 competitor charges as Lloyds
    # -- 26% of all matches -- and wrongly excluded 4,553 RBS borrowers from the
    # prospect population. Those are exactly the poaching targets the lead list wants.
    r"(?<!royal )bank of scotland",      # Bank of Scotland plc (Halifax's legal entity too)
    r"\bhbos\b",                         # HBOS plc
    r"agricultural mortgage corp",       # AMC — Lloyds' farm/agri lender
    r"\bhalifax\b",                      # Halifax (division of Bank of Scotland)
    r"birmingham midshires",             # BM — mortgages
    r"cheltenham (?:& |and )gloucester", # C&G — mortgages
    r"bank of wales",                    # historic BoS brand
    r"black horse",                      # Lloyds motor/asset finance
    r"scottish widows",                  # pensions/insurance
    r"\bmbna\b",                         # credit cards (Lloyds acquired 2017)
]
LLOYDS_RE = re.compile("|".join(LLOYDS_PATTERNS), re.I)

COMPANIES = COMPANIES_CSV
# The 4 columns Stage 3's enrich_sme() returns. NOT the same list as CARRY_COLS in
# Stage 2 (that one is "what survives a rebuild"); these must match enrich_sme's tuple.
STAGE3_COLS = ["lloyds_customer", "recent_charge_status",
               "recent_charge_created_on", "recent_psc_kind"]
# CHARGE_JSON_DIR comes from paths.py — full charge history persisted here
CHARGE_JSON_DIR.mkdir(parents=True, exist_ok=True)

sme = pd.read_csv(COMPANIES, dtype=str, low_memory=False)
sme["is_sme"] = sme["is_sme"].astype(str).str.lower().eq("true")
for col in STAGE3_COLS:
    if col not in sme.columns:
        sme[col] = pd.NA


def enrich_sme(company_num):
    """(lloyds_customer, recent_charge_status, recent_charge_created_on,
        recent_psc_kind) — one paged charges call + one PSC call. Also persists
        the FULL charge list to JSON (free: it's already in memory)."""
    cnum = str(company_num).strip()

    # -- charges: single pass yields BOTH the Lloyds flag and the latest charge --
    lloyds, status, created = "False", None, None
    start, charges = 0, []
    while True:
        r = ch_get(f"/company/{cnum}/charges",
                   params={"items_per_page": 100, "start_index": start})
        if r is None:            lloyds = status = created = "error";  break
        if r.status_code == 404: status = created = "no_charges";      break
        if r.status_code != 200: lloyds = status = created = "error";  break
        data = r.json()
        items = data.get("items", [])
        charges.extend(items)
        for ch in items:
            for p in ch.get("persons_entitled", []):
                if LLOYDS_RE.search(p.get("name", "")):
                    lloyds = "True"        # keep paging: still need the latest charge
        start += len(items)
        if not items or start >= data.get("total_count", 0):
            break
    if status is None:                      # paged through wi thout error
        if charges:
            latest = max(charges, key=lambda c: c.get("created_on") or "")
            status = latest.get("status") or "unknown"
            created = latest.get("created_on") or "unknown"
        else:
            status = created = "no_charges"

    # -- persist the FULL charge list while we hold it (zero extra API calls) --
    # (404/no-charges companies get an empty items list = "pulled, none found",
    #  which lets Stage 4 skip them; error paths write nothing so they retry.)
    if status != "error":
        try:
            (CHARGE_JSON_DIR / f"{cnum}_charges.json").write_text(
                json.dumps({"total_count": len(charges), "items": charges}))
        except OSError:
            pass                            # persistence is best-effort

    # -- PSC: kind of the most recent PSC (latest notified_on) --
    rp = ch_get(f"/company/{cnum}/persons-with-significant-control",
                params={"items_per_page": 100})
    if rp is None:              psc = "error"
    elif rp.status_code == 404: psc = "no_psc"
    elif rp.status_code != 200: psc = "error"
    else:
        items = rp.json().get("items", [])
        if not items:
            psc = "no_psc"
        else:
            latest = max(items, key=lambda it: it.get("notified_on") or "")
            psc = latest.get("kind") or "unknown"

    return lloyds, status, created, psc


# SAFETY: the table now holds non-SME companies too. Every todo selection MUST
# be masked by is_sme or this stage would spend ~154k calls enriching companies
# that were deliberately excluded from the population.
todo = sme.index[sme["is_sme"] & sme[STAGE3_COLS].isna().any(axis=1)]
parallel_fill(sme, todo, enrich_sme, STAGE3_COLS, COMPANIES, desc="sme_enrich")
sme[STAGE3_COLS].apply(lambda c: c.value_counts(dropna=False).head(6))

45 to fetch (of 839394) — 4 keys x 8 workers


sme_enrich: 100%|██████████| 45/45 [00:13<00:00,  3.32it/s]


Saved -> /Users/natchalin_/Projects/final_project/Lloyds/API/CompaniesHouse/company_data/companies.csv


,lloyds_customer,recent_charge_status,recent_charge_created_on,recent_psc_kind
2025-03-28,NaN,NaN,184.0,NaN
2025-03-31,NaN,NaN,151.0,NaN
2025-12-19,NaN,NaN,157.0,NaN
2026-05-29,NaN,NaN,140.0,NaN
False,548290.0,NaN,NaN,NaN
True,19632.0,NaN,NaN,NaN
corporate-entity-person-with-significant-control,NaN,NaN,NaN,44160.0
error,10.0,10.0,NaN,11.0
fully-satisfied,NaN,20275.0,NaN,NaN
individual-person-with-significant-control,NaN,NaN,NaN,502036.0


In [13]:
# --- Stage 4: full charge history -> charges_history.csv -------------------------
# One-time backfill for companies enriched BEFORE Stage 3 started persisting the
# charge JSONs, then a zero-call parse of every JSON into one tidy per-charge table.
# Requires the Stage 3 cell to have run (LLOYDS_RE, CHARGE_JSON_DIR, COMPANIES).
CHARGES_OUT = CHARGES_CSV

sme = pd.read_csv(COMPANIES, dtype=str, low_memory=False)
sme["is_sme"] = sme["is_sme"].astype(str).str.lower().eq("true")
if "n_charges" not in sme.columns:
    sme["n_charges"] = pd.NA


def _charge_file(cnum):
    return CHARGE_JSON_DIR / f"{str(cnum).strip()}_charges.json"


# --- who needs an API call? -------------------------------------------------------
# no_charges/not_found (from stage 3)  -> 0 calls, mark n_charges=0 directly
# JSON already on disk                 -> 0 calls (stage 3 persisted it, or a
#                                         previous stage-4 run did)
# everyone else WITH charges           -> the one-time backfill (~1.8k companies)
no_need   = sme["recent_charge_status"].isin(["no_charges", "not_found"])
unknown   = sme["recent_charge_status"].isna()          # stage 3 not run for these
have_json = sme["com_num"].map(lambda c: _charge_file(c).exists())

sme.loc[no_need & sme["n_charges"].isna(), "n_charges"] = "0"   # free: no call needed

# SAFETY: the table now holds non-SME companies too. Every todo selection MUST
# be masked by is_sme or this stage would spend ~154k calls enriching companies
# that were deliberately excluded from the population.
todo = sme.index[sme["is_sme"] & ~no_need & ~unknown & ~have_json]
if unknown.any():
    print(f"NOTE: {unknown.sum()} companies have no recent_charge_status yet — "
          "run Stage 3 first so they can be classified (skipped here).")
print(f"charge history: {int(no_need.sum())} skipped as no-charges (0 calls) | "
      f"{int((~no_need & have_json).sum())} already on disk (0 calls) | "
      f"{len(todo)} to fetch")


def fetch_charge_history(company_num):
    """Page /charges and persist the FULL item list to JSON. Returns n_charges."""
    cnum = str(company_num).strip()
    start, items = 0, []
    while True:
        r = ch_get(f"/company/{cnum}/charges",
                   params={"items_per_page": 100, "start_index": start})
        if r is None:
            return "error"
        if r.status_code == 404:
            break                                    # no charges registered
        if r.status_code != 200:
            return "error"
        data = r.json()
        page = data.get("items", [])
        items.extend(page)
        start += len(page)
        if not page or start >= data.get("total_count", 0):
            break
    _charge_file(cnum).write_text(json.dumps({"total_count": len(items),
                                              "items": items}))
    return str(len(items))


parallel_fill(sme, todo, fetch_charge_history, "n_charges", COMPANIES, desc="charges_hist")

# --- parse ALL persisted JSONs -> one row per charge (zero API calls) -------------
# Only companies Stage 3 says HAVE charges can contribute rows. The other ~255k have
# an empty JSON on disk, so skipping them avoids ~5x the filesystem work (the real
# cost here is 312k lookups in one folder, not the API).
_worth_parsing = ~sme["recent_charge_status"].isin(["no_charges", "not_found"])

rows = []
for cnum in sme.loc[_worth_parsing, "com_num"]:
    f = _charge_file(cnum)
    if not f.exists():
        continue
    for ch in json.loads(f.read_text()).get("items", []):
        lenders = " | ".join(p.get("name", "") for p in ch.get("persons_entitled", []))
        rows.append({
            "com_num":          cnum,
            "charge_code":      ch.get("charge_code") or ch.get("charge_number"),
            "created_on":       ch.get("created_on"),
            "delivered_on":     ch.get("delivered_on"),
            "satisfied_on":     ch.get("satisfied_on"),
            "status":           ch.get("status"),
            "classification":   (ch.get("classification") or {}).get("description"),
            "persons_entitled": lenders,
            "is_lloyds":        bool(LLOYDS_RE.search(lenders)),
        })
charges_hist = pd.DataFrame(rows)
charges_hist.to_csv(CHARGES_OUT, index=False)

# --- sanity report -----------------------------------------------------------------
n_cos = charges_hist["com_num"].nunique() if len(charges_hist) else 0
n_llo = int(charges_hist["is_lloyds"].sum()) if len(charges_hist) else 0
llo_cos = charges_hist.loc[charges_hist["is_lloyds"], "com_num"].nunique() if len(charges_hist) else 0
print(f"\ncharges_history: {len(charges_hist)} charges across {n_cos} companies "
      f"-> {CHARGES_OUT}")
print(f"  Lloyds-group charges: {n_llo} across {llo_cos} companies "
      f"(cf. lloyds_customer=True in the SME file: "
      f"{(sme['lloyds_customer'] == 'True').sum()})")
if len(charges_hist):
    print(f"  created_on range: {charges_hist['created_on'].min()} -> "
          f"{charges_hist['created_on'].max()}")
    print("  most common lenders:")
    print(charges_hist["persons_entitled"].value_counts().head(8).to_string())

NOTE: 271462 companies have no recent_charge_status yet — run Stage 3 first so they can be classified (skipped here).
charge history: 462905 skipped as no-charges (0 calls) | 105030 already on disk (0 calls) | 0 to fetch
0 to fetch (of 839394) — 4 keys x 8 workers


charges_hist: 0it [00:00, ?it/s]

Nothing to fetch — /Users/natchalin_/Projects/final_project/Lloyds/API/CompaniesHouse/company_data/companies.csv left untouched.



charges_history: 399724 charges across 105023 companies -> /Users/natchalin_/Projects/final_project/Lloyds/API/CompaniesHouse/company_data/charges_history.csv
  Lloyds-group charges: 39887 across 14862 companies (cf. lloyds_customer=True in the SME file: 19632)
  created_on range: 1898-07-08 -> 2026-08-14
  most common lenders:
persons_entitled
National Westminster Bank PLC          32608
Barclays Bank PLC                      27389
Lloyds Bank PLC                        18587
Hsbc Bank PLC                          13050
The Royal Bank of Scotland PLC         12533
The Mortgage Works (UK) PLC            11966
Together Commercial Finance Limited    10842
Paragon Bank PLC                        9305


In [14]:
# --- Stage 4b: rebuild charges_history.csv from disk (no API calls) --------------
# Parses every persisted charge JSON for companies in the CURRENT SME file.
# Run after any population expansion so the label source stays complete.
import json
import pandas as pd
from pathlib import Path

sme = pd.read_csv(COMPANIES, dtype=str, low_memory=False)
sme["is_sme"] = sme["is_sme"].astype(str).str.lower().eq("true")
before = pd.read_csv(CHARGES_OUT, dtype=str) if Path(CHARGES_OUT).exists() else pd.DataFrame()

# skip the ~255k companies Stage 3 marked as having no charges: their JSONs are
# empty, and touching 312k files in one folder is what makes this slow.
_worth_parsing = ~sme["recent_charge_status"].isin(["no_charges", "not_found"])
print(f"parsing {_worth_parsing.sum():,} of {len(sme):,} companies "
      f"(skipping {(~_worth_parsing).sum():,} with no charges)")

rows, missing_json = [], 0
for cnum in sme.loc[_worth_parsing, "com_num"]:
    f = _charge_file(cnum)
    if not f.exists():
        missing_json += 1
        continue
    for ch in json.loads(f.read_text()).get("items", []):
        lenders = " | ".join(p.get("name", "") for p in ch.get("persons_entitled", []))
        rows.append({
            "com_num":          cnum,
            "charge_code":      ch.get("charge_code") or ch.get("charge_number"),
            "created_on":       ch.get("created_on"),
            "delivered_on":     ch.get("delivered_on"),
            "satisfied_on":     ch.get("satisfied_on"),
            "status":           ch.get("status"),
            "classification":   (ch.get("classification") or {}).get("description"),
            "persons_entitled": lenders,
            "is_lloyds":        bool(LLOYDS_RE.search(lenders)),
        })

charges_hist = pd.DataFrame(rows)
charges_hist.to_csv(CHARGES_OUT, index=False)

# --- before/after report ----------------------------------------------------------
def _stats(df):
    if not len(df):
        return 0, 0, 0
    ll = df["is_lloyds"].astype(str).str.lower().eq("true")
    return len(df), df["com_num"].nunique(), df.loc[ll, "com_num"].nunique()

b_rows, b_firms, b_ll = _stats(before)
a_rows, a_firms, a_ll = _stats(charges_hist)
print(f"charges_history rebuilt -> {CHARGES_OUT}")
print(f"  rows          : {b_rows:>8,}  ->  {a_rows:>8,}   ({a_rows-b_rows:+,})")
print(f"  firms         : {b_firms:>8,}  ->  {a_firms:>8,}   ({a_firms-b_firms:+,})")
print(f"  Lloyds firms  : {b_ll:>8,}  ->  {a_ll:>8,}   ({a_ll-b_ll:+,})   <- the label")
print(f"  companies with no JSON on disk (never fetched): {missing_json:,}")

# cross-check against the Stage-3 flag; these two should now agree closely
flag = sme["lloyds_customer"].astype(str).eq("True").sum()
print(f"\n  Stage-3 lloyds_customer=True : {flag:,}")
print(f"  charges_history Lloyds firms : {a_ll:,}   (gap {flag - a_ll:+,})")

parsing 376,489 of 839,394 companies (skipping 462,905 with no charges)
charges_history rebuilt -> /Users/natchalin_/Projects/final_project/Lloyds/API/CompaniesHouse/company_data/charges_history.csv
  rows          :  399,724  ->   399,724   (+0)
  firms         :  105,023  ->   105,023   (+0)
  Lloyds firms  :   14,862  ->    14,862   (+0)   <- the label
  companies with no JSON on disk (never fetched): 271,459

  Stage-3 lloyds_customer=True : 19,632
  charges_history Lloyds firms : 14,862   (gap +4,770)


In [14]:
# --- Stage 5: backfill panel fields from the profile (one-time, resumable) -------
# Only SME rows missing date_of_creation are fetched; new companies get these at
# Stage 1 for free, so this backfill shrinks to zero over time.
COMPANIES = COMPANIES_CSV
PANEL_COLS = ["date_of_creation", "company_status",
              "last_accounts_made_up_to", "next_accounts_due_on"]

sme = pd.read_csv(COMPANIES, dtype=str, low_memory=False)
sme["is_sme"] = sme["is_sme"].astype(str).str.lower().eq("true")
for c in PANEL_COLS:
    if c not in sme.columns:
        sme[c] = pd.NA


def fetch_panel_fields(company_num):
    """(date_of_creation, company_status, last_accounts_made_up_to,
        next_accounts_due_on) from ONE profile call."""
    r = ch_get(f"/company/{str(company_num).strip()}")
    if r is None:              return ("error",) * 4
    if r.status_code == 404:   return ("not_found",) * 4
    if r.status_code != 200:   return ("error",) * 4
    j = r.json()
    acc = j.get("accounts", {})
    return (j.get("date_of_creation") or "unknown",
            j.get("company_status") or "unknown",
            acc.get("last_accounts", {}).get("made_up_to") or "none",
            acc.get("next_accounts", {}).get("due_on") or "none")


# SAFETY: the table now holds non-SME companies too — mask by is_sme or this
# stage would spend calls on companies excluded from the population.
todo = sme.index[sme["is_sme"] & sme["date_of_creation"].isna()]
parallel_fill(sme, todo, fetch_panel_fields, PANEL_COLS, COMPANIES, desc="panel_fields")

# --- sanity report ----------------------------------------------------------------
print("\npanel-field coverage:")
for c in PANEL_COLS:
    filled = sme[c].notna() & ~sme[c].isin(["error", "not_found"])
    print(f"  {c:26s} {int(filled.sum())}/{len(sme)}")
if sme["date_of_creation"].notna().any():
    ok = sme.loc[~sme["date_of_creation"].isin(["error", "not_found", "unknown"]),
                 "date_of_creation"]
    print(f"\n  incorporation dates: {ok.min()} -> {ok.max()}")
    print("  company_status:")
    print(sme["company_status"].value_counts(dropna=False).head(6).to_string())

0 to fetch (of 839344) — 4 keys x 8 workers


panel_fields: 0it [00:00, ?it/s]

Nothing to fetch — /Users/natchalin_/Projects/final_project/Lloyds/API/CompaniesHouse/company_data/companies.csv left untouched.

panel-field coverage:
  date_of_creation           839316/839344
  company_status             839316/839344
  last_accounts_made_up_to   839316/839344
  next_accounts_due_on       839316/839344

  incorporation dates: 1856-09-25 -> 2026-05-31
  company_status:
company_status
active            838376
liquidation          679
receivership         145
administration        64
dissolved             44
error                 27


In [ ]:
# --- Stage 6: full filing history -> filings_history.csv --------------------------
import numpy as np
# One /filing-history call per company (paged), persisted to JSON, then parsed to a
# per-filing table. Requires the client cell (ch_get, parallel_fill) + COMPANIES.
FILING_JSON_DIR = CHARGE_JSON_DIR                       # reuse the same json folder
FILINGS_OUT     = FILINGS_CSV

# PULL MODE — three settings, in the order you should use them:
#
#   "stratified"  EVERY known Lloyds customer + N_CONTROLS random non-customers.
#                 ~76 min for 30k companies. This is the one to run FIRST.
#                 Why: all the information about "what predicts conversion" lives in
#                 the ~10k customers. A plain first-N or random probe of a few hundred
#                 lands ~16 customers, which can only detect a ~35 percentage-point
#                 effect — useless. Taking every customer gives +-0.5% precision, so a
#                 real effect is unmissable. And the pull is NOT throwaway: these are
#                 exactly the rows a case-control model trains on (same design as
#                 data_conditional.ipynb).
#
#   "full"        all ~312k companies, ~13h. Needed only to SCORE the live lead list,
#                 not to decide whether a feature works. Resumable — it skips anyone
#                 already on disk, so "stratified" then "full" costs nothing extra.
#
#   "first_n"     legacy first-N-rows behaviour. Do NOT use for evaluation: the SME
#                 file is name-sorted, so the first rows are ~1% customers vs 3.3%
#                 overall (lots of numeric-named RTM shells that never borrow).
PULL_MODE   = "stratified"
N_CONTROLS  = 20000        # non-customers sampled alongside every customer
SAMPLE_N    = 500          # only used when PULL_MODE == "first_n"
SEED        = 42

sme = pd.read_csv(COMPANIES, dtype=str, low_memory=False)
sme["is_sme"] = sme["is_sme"].astype(str).str.lower().eq("true")
if "n_filings" not in sme.columns:
    sme["n_filings"] = pd.NA


def _filing_file(cnum):
    return FILING_JSON_DIR / f"{str(cnum).strip()}_filings.json"


def fetch_filing_history(company_num):
    """Page /filing-history and persist the FULL item list to JSON. Returns n_filings."""
    cnum = str(company_num).strip()
    start, items = 0, []
    while True:
        r = ch_get(f"/company/{cnum}/filing-history",
                   params={"items_per_page": 100, "start_index": start})
        if r is None:
            return "error"
        if r.status_code == 404:
            break                                      # no filing history
        if r.status_code != 200:
            return "error"
        data = r.json()
        page = data.get("items", [])
        items.extend(page)
        start += len(page)
        if not page or start >= data.get("total_count", 0):
            break
    _filing_file(cnum).write_text(json.dumps({"total_count": len(items), "items": items}))
    return str(len(items))


# who needs a call? anyone without a persisted JSON and without n_filings yet
have_json = sme["com_num"].map(lambda c: _filing_file(c).exists())
# SAFETY: the table now holds non-SME companies too. Every todo selection MUST
# be masked by is_sme or this stage would spend ~154k calls enriching companies
# that were deliberately excluded from the population.
todo = sme.index[sme["is_sme"] & sme["n_filings"].isna() & ~have_json]

_is_cust = sme["lloyds_customer"].astype(str).str.lower().eq("true")

if PULL_MODE == "stratified":
    _rng   = np.random.default_rng(SEED)
    _pos   = sme.index[_is_cust]                                  # every known customer
    _negs  = sme.index[~_is_cust]
    _neg   = pd.Index(_rng.choice(_negs, size=min(N_CONTROLS, len(_negs)), replace=False))
    todo   = todo.intersection(_pos.union(_neg))
    _npos  = int(_is_cust.loc[todo].sum())
    print(f"STRATIFIED: {len(todo):,} companies to fetch "
          f"({_npos:,} Lloyds customers + {len(todo)-_npos:,} controls) "
          f"~{len(todo) / (len(_LANES) / _MIN_INTERVAL) / 60:.0f} min")
elif PULL_MODE == "full":
    print(f"FULL PULL: {len(todo):,} companies to fetch "
          f"(~{len(todo) / (len(_LANES) / _MIN_INTERVAL) / 3600:.1f}h, resumable).")
elif PULL_MODE == "first_n":
    todo = todo[:SAMPLE_N]
    print(f"FIRST-N (biased — evaluation-unsafe): {len(todo):,} companies.")
else:
    raise ValueError("PULL_MODE must be 'stratified', 'full' or 'first_n'")

parallel_fill(sme, todo, fetch_filing_history, "n_filings", COMPANIES, desc="filing_hist")

# --- parse ALL persisted filing JSONs -> one row per filing (zero API calls) ------
rows = []
for cnum in sme.loc[sme["is_sme"], "com_num"]:
    f = _filing_file(cnum)
    if not f.exists():
        continue
    for it in json.loads(f.read_text()).get("items", []):
        dv = it.get("description_values") or {}
        rows.append({
            "com_num":      cnum,
            "category":     it.get("category"),
            "type":         it.get("type"),
            "date":         it.get("date"),                       # date filed/received
            "made_up_date": dv.get("made_up_date") or it.get("action_date"),
            "paper_filed":  bool(it.get("paper_filed")),
            "doc_metadata": (it.get("links") or {}).get("document_metadata"),  # for Phase B iXBRL
        })
filings_hist = pd.DataFrame(rows)
filings_hist.to_csv(FILINGS_OUT, index=False)

# --- sanity report -----------------------------------------------------------------
n_cos = filings_hist["com_num"].nunique() if len(filings_hist) else 0
acc = (filings_hist[filings_hist["category"] == "accounts"]
       if len(filings_hist) else filings_hist)
print(f"\nfilings_history: {len(filings_hist)} filings across {n_cos} companies -> {FILINGS_OUT}")
print(f"  accounts filings: {len(acc)} across "
      f"{acc['com_num'].nunique() if len(acc) else 0} companies")
if len(filings_hist):
    print("  categories:")
    print(filings_hist["category"].value_counts().head(8).to_string())

# --- is the pull informative yet? (customers vs controls actually fetched) --------
_pulled = set(filings_hist["com_num"]) if len(filings_hist) else set()
_have   = sme[sme["com_num"].isin(_pulled)]
_hp     = _have["lloyds_customer"].astype(str).str.lower().eq("true").sum()
print(f"\ncoverage so far: {len(_have):,} companies with filing history "
      f"({_hp:,} Lloyds customers, {len(_have)-_hp:,} controls)")
if _hp < 500:
    print("  ⚠️  under ~500 customers: too few to judge a feature "
          "(see the PULL_MODE notes). Run PULL_MODE='stratified'.")
else:
    import math
    _se = math.sqrt(0.3 * 0.7 / _hp)
    print(f"  precision on a ~30%-prevalence feature: +-{_se*100:.1f}% "
          f"-> can detect gaps larger than ~{3*_se*100:.1f} percentage points")

STRATIFIED: 12,310 companies to fetch (0 Lloyds customers + 12,310 controls) ~31 min
12310 to fetch (of 839344) — 4 keys x 8 workers


filing_hist:   2%|▏         | 200/12310 [00:36<1:07:46,  2.98it/s]

  200/12310 | 5.53/s | ETA ~37 min | checkpoint saved


filing_hist:   2%|▏         | 295/12310 [00:49<33:22,  6.00it/s]  


In [15]:
# --- Analysis-ready load: consistent dtypes & safe comparisons -----------------
# The CSV stores everything as text (uniform "True"/"False"/sentinels). This loads
# a typed view where the boolean flags become real nullable booleans, so you can do
# `sme_typed[sme_typed.lloyds_customer]` or `== True`. Non True/False values
# (e.g. 'error') become <NA> and are reported so nothing is hidden.
COMPANIES = COMPANIES_CSV
_all_typed = pd.read_csv(COMPANIES, dtype=str, low_memory=False)

# The table holds non-SME companies too (the do-not-re-pull cache), so narrow to the
# modelling population — `sme_typed` should mean what its name says.
_is_sme   = _all_typed["is_sme"].astype(str).str.lower().eq("true")
sme_typed = _all_typed[_is_sme].copy()
print(f"loaded {len(_all_typed):,} rows -> {len(sme_typed):,} is_sme rows "
      f"({int((~_is_sme).sum()):,} non-SME cached rows held back)")

bool_map = {"True": True, "False": False}
for col in ("lloyds_customer", "accounts_overdue"):
    if col in sme_typed.columns:
        odd = sme_typed.loc[sme_typed[col].notna() & ~sme_typed[col].isin(bool_map), col]
        if len(odd):
            print(f"Non True/False {col} values -> <NA>:")
            print(odd.value_counts().to_string(), "\n")
        sme_typed[col] = sme_typed[col].map(bool_map).astype("boolean")
        print(f"{col}:")
        print(sme_typed[col].value_counts(dropna=False).to_string(), "\n")

for col in ("recent_psc_kind", "recent_charge_status"):
    if col in sme_typed.columns:
        print(f"{col}:")
        print(sme_typed[col].value_counts(dropna=False).to_string(), "\n")

# `sme_typed` is analysis-ready (e.g. sme_typed[sme_typed.lloyds_customer]).
# The CSV on disk stays canonical text.
sme_typed.head()


loaded 839,344 rows -> 567,883 is_sme rows (271,461 non-SME cached rows held back)
Non True/False lloyds_customer values -> <NA>:
lloyds_customer
error    10 

lloyds_customer:
lloyds_customer
False    548243
True      19630
<NA>         10 

Non True/False accounts_overdue values -> <NA>:
accounts_overdue
unknown    21 

accounts_overdue:
accounts_overdue
False    559132
True       8730
<NA>         21 

recent_psc_kind:
recent_psc_kind
individual-person-with-significant-control          502000
corporate-entity-person-with-significant-control     44154
no_psc                                               21102
legal-person-person-with-significant-control           608
error                                                   11
super-secure-person-with-significant-control             8 

recent_charge_status:
recent_charge_status
no_charges         462864
outstanding         84692
fully-satisfied     20275
part-satisfied         40
error                  10
satisfied               2 



,com_num,name,sic_code,account_type,is_sme,accounts_overdue,date_of_creation,company_status,last_accounts_made_up_to,next_accounts_due_on,...,com_name,lloyds_customer,recent_psc_kind,recent_charge_status,recent_charge_created_on,n_charges,n_filings,region,town_api,Unnamed: 0
12,13172868,369 TECHNICAL LTD,28302,total-exemption-full,True,False,2021-02-02,active,2025-02-28,2026-11-30,...,NaN,False,individual-person-with-significant-control,no_charges,no_charges,0,10,Wales,NaN,NaN
15,07852962,369 UPLAND ROAD RTM COMPANY LIMITED,98000,micro-entity,True,False,2011-11-18,active,2025-09-28,2027-06-28,...,NaN,False,no_psc,no_charges,no_charges,0,45,London,NaN,NaN
16,14568066,369 VORTEX LIMITED,64209,micro-entity,True,False,2023-01-03,active,2026-01-31,2027-10-31,...,NaN,False,individual-person-with-significant-control,no_charges,no_charges,0,7,East Midlands,NaN,NaN
17,15623849,369 WATCH PARTS LTD,47990,total-exemption-full,True,False,2024-04-07,active,2025-04-30,2027-01-31,...,NaN,False,individual-person-with-significant-control,no_charges,no_charges,0,4,London,NaN,NaN
20,16023853,369MULTI LTD,47520,micro-entity,True,False,2024-10-17,active,2025-10-31,2027-07-31,...,NaN,False,individual-person-with-significant-control,no_charges,no_charges,0,5,North West,NaN,NaN


In [16]:
# --- Repair com_num: realign leading zeros to the raw originals ------------------
# UK company numbers keep leading zeros (e.g. 05914136). Opening the CSV in Excel
# can silently drop them (05914136 -> 5914136, or float-ify to 5914136.0). This
# remaps com_num back to the canonical values in company_raw/. Idempotent:
# reports 0 changed when the file is already aligned.
RAW_DIR = CH_DATA / "company_raw"
COMPANIES = COMPANIES_CSV


def _norm(x):
    """Collapse a com_num to a match key: drop a stray '.0', then leading zeros."""
    x = str(x).strip()
    if x.endswith(".0"):
        x = x[:-2]
    return x.lstrip("0")


# canonical map: normalized form -> original com_num (from the raw files)
raw = pd.concat([pd.read_csv(f, dtype=str) for f in sorted(RAW_DIR.glob("*.csv"))],
                ignore_index=True).drop_duplicates("com_num")
canon = {_norm(c): c for c in raw["com_num"]}

sme = pd.read_csv(COMPANIES, dtype=str)
before = sme["com_num"].copy()
sme["com_num"] = sme["com_num"].map(lambda x: canon.get(_norm(x), str(x).strip()))

changed = int((before != sme["com_num"]).sum())
unmatched = sme.loc[~sme["com_num"].isin(set(raw["com_num"])), "com_num"]
sme.to_csv(COMPANIES, index=False)
print(f"Aligned com_num -> {COMPANIES}")
print(f"  rows changed: {changed}")
print(f"  not matching any raw com_num: {len(unmatched)}")
if len(unmatched):
    print("  examples:", unmatched.head(5).tolist())


Aligned com_num -> /Users/natchalin_/Projects/final_project/Lloyds/API/CompaniesHouse/company_data/companies.csv
  rows changed: 1
  not matching any raw com_num: 690793
  examples: ['06369946', '06381193', '07218758', '07806564', '08461200']
